## Train a CLIP model on the generated geometric-shape images.
## This script assumes that the geometric-shape images have already been generated and stored in the "data/SHAPE" directory.

## Import necessary libraries for training the CLIP model.

In [ ]:
from PIL import Image
import csv
import torch
import torch.nn.functional as F
from pathlib import Path
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
from CLIP_model_utils import TokenizeCaption
from CLIP_model import Config, CLIP

## Initialize the CLIP model, criterion, optimizer, and device.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = TokenizeCaption()
tokenizer.build_vocab(Path(__file__).resolve().parent / "data" / "SHAPE" / "metadata.csv")

## Dataset and dataloader for the geometric-shape images

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, csv_file, image_folder):
        self.csv_file = csv_file
        self.image_folder = image_folder
        self.tokenizer = tokenizer
        self.dataset = []
        with open(csv_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                image_name = row["image"]
                caption = row["caption"]
                image_path = os.path.join(image_folder, image_name)
                self.dataset.append((image_path, caption))
        self.transform = transforms.Compose([
            transforms.Resize((Config.image_resolution, Config.image_resolution)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image_path, caption = self.dataset[idx]
        from PIL import Image
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        # caption to tensor as word indices
        caption = torch.tensor(self.tokenizer.encode_caption(caption), dtype=torch.long)
        
        return image, caption

model = CLIP(Config.image_resolution, len(tokenizer.vocab), Config.hidden_dim, Config.output_dim).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

dataset = ImageTextDataset(csv_file=Path(__file__).resolve().parent / "data" / "SHAPE" / "metadata.csv", image_folder=Path(__file__).resolve().parent / "data" / "SHAPE")
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

## Loss

In [ ]:
# Symmetric contrastive loss
def clip_loss(logits_per_image, logits_per_text):
    batch_size = logits_per_image.size(0)

    # Positive pair i is image[i] with text[i].
    targets = torch.arange(
        batch_size, device=logits_per_image.device
    )

    loss_i = F.cross_entropy(logits_per_image, targets)
    loss_t = F.cross_entropy(logits_per_text, targets)

    return (loss_i + loss_t) / 2

## Start training

In [ ]:
writer = SummaryWriter()
num_epochs = 1  # Set the number of training epochs

# Loop training for the CLIP model.
for epoch in range(num_epochs):
    model.train()
    for images, captions in dataloader:
        # Move the image and label to the appropriate device (CPU or GPU)
        # Forward pass through the CLIP model
        images = images.to(device)
        captions = captions.to(device)
        logits_per_image, logits_per_text = model(images, captions)

        # Compute the loss and perform backpropagation
        loss = clip_loss(logits_per_image, logits_per_text)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
    writer.add_scalar("Loss/train", loss.item(), epoch)
    # Save the model checkpoint after 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = Path(__file__).resolve().parent / "checkpoints"
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        checkpoint_path = checkpoint_path / f"clip_model_epoch_{epoch+1}.pth"
        torch.save(model.state_dict(), checkpoint_path)
        writer.add_text("Checkpoints", f"Saved checkpoint at epoch {epoch+1}", epoch)

writer.close()